# 06 - RF SMOTE Current Stress

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import importlib.util
import json
import joblib

import mlflow
import mlflow.sklearn
import pandas as pd
from mlflow.models import infer_signature
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, StackingClassifier

CANDIDATE_DIRS = [
    Path.cwd(),
    Path.cwd() / "nostressia-machine-learning" / "Current-Stress" / "notebooks" / "experiments",
]
UTILS_PATH = next((d / "mlflow_utils.py" for d in CANDIDATE_DIRS if (d / "mlflow_utils.py").exists()), None)
if UTILS_PATH is None:
    raise FileNotFoundError("mlflow_utils.py tidak ditemukan. Pastikan notebook dijalankan dari root repo atau folder experiments.")

spec = importlib.util.spec_from_file_location("cs_mlflow_utils", UTILS_PATH)
cs_utils = importlib.util.module_from_spec(spec)
assert spec.loader is not None
spec.loader.exec_module(cs_utils)

RANDOM_STATE = cs_utils.RANDOM_STATE
configure_mlflow = cs_utils.configure_mlflow
set_seeds = cs_utils.set_seeds
load_current_stress_dataset = cs_utils.load_current_stress_dataset
get_dataset_path = cs_utils.get_dataset_path
select_feature_set = cs_utils.select_feature_set
split_data = cs_utils.split_data
build_preprocessor = cs_utils.build_preprocessor
evaluate_classification = cs_utils.evaluate_classification
measure_latency_percentiles = cs_utils.measure_latency_percentiles
log_classification_artifacts = cs_utils.log_classification_artifacts
log_run_metadata = cs_utils.log_run_metadata
create_local_output_dir = cs_utils.create_local_output_dir

repo_root = configure_mlflow()
set_seeds(RANDOM_STATE)

from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline


In [ ]:
NOTEBOOK_NAME = "06_rf_smote_current_stress.ipynb"
RUN_NAME = "RF + SMOTE - Current Stress"
FEATURE_SET = "all"
REGISTERED_MODEL_NAME = "CurrentStress"

raw_df, feature_df, y = load_current_stress_dataset(repo_root)
dataset_source = str(get_dataset_path(repo_root))
X = select_feature_set(feature_df, FEATURE_SET)
X_train, X_test, y_train, y_test = split_data(X, y)

num_cols = X_train.select_dtypes(include=["number"]).columns.tolist()
cat_cols = X_train.select_dtypes(exclude=["number"]).columns.tolist()
preprocessor = build_preprocessor(num_cols, cat_cols)

model = ImbPipeline(steps=[("preprocessor", preprocessor),("smote", SMOTE(random_state=RANDOM_STATE)),("classifier", RandomForestClassifier(random_state=RANDOM_STATE, n_estimators=300, n_jobs=-1, class_weight="balanced"))])

with mlflow.start_run(run_name=RUN_NAME) as run:
    dataset_payload = pd.concat([X_train.assign(target=y_train.values, split_set="train"),X_test.assign(target=y_test.values, split_set="test")], ignore_index=True)
    log_run_metadata(
        run_description="RF + SMOTE; features=all; dataset=current_stress_v1; split=80/20; random_state=42",
        tags={"features": FEATURE_SET, "model": RUN_NAME},
        params={"registered_model_name": REGISTERED_MODEL_NAME, "model_type":"RandomForestClassifier", "balancing":"SMOTE"},
        dataset_df=dataset_payload,
        dataset_context="training",
        dataset_source=dataset_source,
    )
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    cv_scores = cross_val_score(model, X_train, y_train, cv=cv, scoring="f1_weighted", n_jobs=-1)
    mlflow.log_metric("cv_f1_weighted_mean", float(cv_scores.mean()))
    mlflow.log_metric("cv_f1_weighted_std", float(cv_scores.std()))

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test) if hasattr(model, "predict_proba") else None
    metrics = evaluate_classification(y_test, y_pred, y_proba)
    latency_metrics = measure_latency_percentiles(model, X_test, include_predict_proba=y_proba is not None)
    metrics.update(latency_metrics)
    mlflow.log_metrics(metrics)

    local_output_dir = create_local_output_dir(repo_root, NOTEBOOK_NAME, run.info.run_id)
    print(f"[ipynb-result] Output folder: {local_output_dir}")
    log_classification_artifacts(y_test, y_pred, local_output_dir, y_proba, prefix="test")
    (local_output_dir / "metrics.json").write_text(json.dumps(metrics, indent=2), encoding="utf-8")
    joblib.dump(model, local_output_dir / "model.joblib")

    mlflow.log_artifacts(str(local_output_dir), artifact_path="local_outputs")
    signature = infer_signature(X_train, model.predict(X_train))
    mlflow.sklearn.log_model(sk_model=model,artifact_path="model",signature=signature,input_example=X_train.head(5),registered_model_name=REGISTERED_MODEL_NAME)
    print("Run ID:", run.info.run_id)
    print("Local output:", local_output_dir)
    print("Registered model:", REGISTERED_MODEL_NAME)
    print(pd.Series(metrics).sort_index())
